In [2]:
!pip install arxiv

In [3]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


C:\Users\Abhijeet\AppData\Local\Temp\ipykernel_25768\2616709328.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [4]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [5]:
wiki.name

'wikipedia'

In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)

vectordb = FAISS.from_documents(documents=documents,embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))
retriever = vectordb.as_retriever()

retriever


USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DC58BA7620>, search_kwargs={})

In [7]:
# Create the Retriever Tool
from langchain_core.tools.retriever import create_retriever_tool

web_retriever_tool = create_retriever_tool(retriever,"langsmith_search",
"Search for information about LangSmith for any questions about langSmith .you must use this tool!")



In [8]:
web_retriever_tool.name

'langsmith_search'

In [9]:
# Arxiv Tool 
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

arxiv_tool.name

'arxiv'

In [10]:
tools = [wiki,web_retriever_tool,arxiv_tool]

In [11]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Abhijeet\\Desktop\\ai-engineer\\langchain\\langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith for any questions about langSmith .you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001DC59BF54E0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001DC822C5D20>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_avail

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()


llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")
print("LLM Successfully Initialized")

LLM Successfully Initialized


In [13]:
from langchain_classic import hub
from langsmith import Client
from dotenv import load_dotenv

load_dotenv()

client = Client()
# 🛡️ Pass the required security flag inside the pull function
prompt = client.pull_prompt("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)
print('Inside the Hub Prompt ')
prompt.messages

Inside the Hub Prompt 


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [14]:
# agents 's core idea is to use a language Model to choose a sequence of actions to take in chains , a sequence of actions is hardcoded . 
# In agents , a language model is used as a reasoning engine to determine which actions to take and in which order ==> so the order in which tool are defined is what matters

from langchain.agents import create_agent

system_prompt = "You are a helpfu; assistant . Use your tools to answer questions accurately"
agent =  create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,

)

print("Agent Successfully Initialized")

Agent Successfully Initialized


In [ ]:
## Agent Executor

import streamlit as st

st.title("Multi Source RAG Pipeline Chatbot")

input_text = st.write("Type your Questiosn")


response =  agent.invoke({
    "messages": [
        {"role": "user","content":"Who is A.P.J Abdul Kalam"}
    
    ]
})

# 1. Grab the final AIMessage block array from the graph path
final_message = response["messages"][-1]

# 2. Extract the inner text block from the modern list layout smoothly
if isinstance(final_message.content, list):
    final_answer = final_message.content[0]['text']
else:
    final_answer = final_message.content

print("\n--- Final Agent Response ---")
print(final_answer)





c:\Users\Abhijeet\Desktop\ai-engineer\langchain\langchain\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3102: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


ValueError: contents are required.